In [1]:
import pandas as pd

In [3]:
df_gmv = pd.read_csv("./cleaned_data/SM_HN_HCM_REV_filtered_2026_5.csv", encoding='utf-8-sig')
df_leads = pd.read_csv("./cleaned_data/PalFish - Chatpage 2026 - Trực page T01-T05_2026.csv", encoding='utf-8-sig')


### Thống kê sdt của gmv và lead

In [5]:
# Hàm đếm độ dài số điện thoại
def phone_length_stats(df, phone_col):
    # Chuyển về string
    phones = (
        df[phone_col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Chỉ giữ lại số
    phones = phones.str.replace(r"\D", "", regex=True)

    # Tính độ dài
    lengths = phones.str.len()

    # Đếm tần suất
    stats = (
        lengths.value_counts()
        .sort_index()
        .rename_axis("Phone Length")
        .reset_index(name="Count")
    )

    return stats

# GMV
print("===== GMV =====")
print(phone_length_stats(df_gmv, "Phone"))

# Leads
print("\n===== LEADS =====")
print(phone_length_stats(df_leads, "Số điện thoại"))

===== GMV =====
   Phone Length  Count
0            11    345
1            12    101
2            13     17

===== LEADS =====
   Phone Length  Count
0             0    229
1             4      1
2            10      9
3            11  14372
4            12   1560
5            13    160
6            14      3
7            15      2


In [8]:
def phone_length_examples(df, phone_col, n=5):
    temp = pd.DataFrame({
        "Raw": df[phone_col].fillna("").astype(str)
    })

    # Chỉ dùng để tính độ dài
    temp["Clean"] = (
        temp["Raw"]
        .str.strip()
        .str.replace(r"\D", "", regex=True)
    )

    temp["Length"] = temp["Clean"].str.len()

    for length in sorted(temp["Length"].unique()):
        print(f"\n===== Length = {length} =====")

        samples = (
            temp[temp["Length"] == length]
            [["Raw", "Clean"]]
            .drop_duplicates()
            .head(n)
        )

        print(samples.to_string(index=False))
        
print("===== GMV =====")
phone_length_examples(df_gmv, "Phone")

print("\n===== LEADS =====")
phone_length_examples(df_leads, "Số điện thoại")

===== GMV =====

===== Length = 11 =====
         Raw       Clean
84-908224672 84908224672
84-909517679 84909517679
84-964678701 84964678701
84-389970625 84389970625
84-938593961 84938593961

===== Length = 12 =====
          Raw        Clean
420-734602066 420734602066
81-7031882708 817031882708
81-8035704573 818035704573
81-7085394685 817085394685
81-8051986529 818051986529

===== Length = 13 =====
           Raw         Clean
49-15750418390 4915750418390
49-17644461422 4917644461422
49-15205315239 4915205315239
49-17641893888 4917641893888
49-17687465203 4917687465203

===== LEADS =====

===== Length = 0 =====
    Raw Clean
             
QR ZALO      
      -      
  MÃ QR      
QR Zalo      

===== Length = 4 =====
  Raw Clean
16:33  1633

===== Length = 10 =====
        Raw      Clean
82-66831308 8266831308
84-35449883 8435449883
84-77705527 8477705527
82-66248699 8266248699
84-56562448 8456562448

===== Length = 11 =====
         Raw       Clean
84-919249237 84919249237
84-9654685

### Thống kê uid của gmv và leads

In [6]:
def uid_length_stats(df, uid_col):
    # Chuẩn hóa UID
    uids = (
        df[uid_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()      # hoặc .str.lower()
    )

    # Tính độ dài
    lengths = uids.str.len()

    # Thống kê
    stats = (
        lengths.value_counts()
        .sort_index()
        .rename_axis("UID Length")
        .reset_index(name="Count")
    )

    return stats


# GMV
print("===== GMV UID =====")
print(uid_length_stats(df_gmv, "UID"))

# Leads
print("\n===== LEADS UID =====")
print(uid_length_stats(df_leads, "UID"))

===== GMV UID =====
   UID Length  Count
0          10    463

===== LEADS UID =====
   UID Length  Count
0           0    761
1           1      1
2           2      1
3           9      4
4          10   8759
5          11      5
6          12   6805


In [7]:
def uid_length_examples(df, uid_col, n=5):
    uids = (
        df[uid_col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    temp = pd.DataFrame({
        "UID": uids,
        "Length": uids.str.len()
    })

    for length in sorted(temp["Length"].unique()):
        print(f"\n===== Length = {length} =====")
        print(
            temp[temp["Length"] == length]["UID"]
            .drop_duplicates()
            .head(n)
            .tolist()
        )

print("GMV")
uid_length_examples(df_gmv, "UID")

print("\nLEADS")
uid_length_examples(df_leads, "UID")

GMV

===== Length = 10 =====
['3311001921', '3179205818', '3311304274', '3310902627', '3309271098']

LEADS

===== Length = 0 =====
['']

===== Length = 1 =====
['`']

===== Length = 2 =====
['L8']

===== Length = 9 =====
['309845189', '310072108', '307486256', '307272816']

===== Length = 10 =====
['3309599159', '3309599173', '3309599181', '3309622104', '3288358317']

===== Length = 11 =====
['310991875.0', '305383787.0', '158072687.0', '305580541.0', '305825571.0']

===== Length = 12 =====
['3310947243.0', '3310952712.0', '3293618952.0', '3310947249.0', '3310952706.0']


### Clean sdt và phone của cả 2


In [9]:
import pandas as pd
import numpy as np

# ==========================
# CONFIG
# ==========================

GMV_FILE = "./cleaned_data/SM_HN_HCM_REV_filtered_2026_5.csv"
LEADS_FILE = "./cleaned_data/PalFish - Chatpage 2026 - Trực page T01-T05_2026.csv"

GMV_PHONE_COL = "Phone"
GMV_UID_COL = "UID"

LEADS_PHONE_COL = "Số điện thoại"
LEADS_UID_COL = "UID"


# ==========================
# CLEAN UID
# ==========================

def clean_uid(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\.0$", "", regex=True)
    )

    s = s.replace("", np.nan)

    return s


# ==========================
# CLEAN PHONE
# ==========================

def clean_phone(series):
    s = (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\D", "", regex=True)
    )

    # quá ngắn hoặc quá dài -> invalid
    s = s.where(s.str.len().between(9, 13), np.nan)

    return s


# ==========================
# LOAD
# ==========================

df_gmv = pd.read_csv(GMV_FILE)
df_leads = pd.read_csv(LEADS_FILE)


# ==========================
# CLEAN
# ==========================

df_gmv["UID_clean"] = clean_uid(df_gmv[GMV_UID_COL])
df_gmv["Phone_clean"] = clean_phone(df_gmv[GMV_PHONE_COL])

df_leads["UID_clean"] = clean_uid(df_leads[LEADS_UID_COL])
df_leads["Phone_clean"] = clean_phone(df_leads[LEADS_PHONE_COL])


# ==========================
# EXPORT
# ==========================

df_gmv.to_csv(
    "gmv_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_leads.to_csv(
    "leads_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done!")

Done!


### Check douplicate uid trong gmv


In [79]:
df_gmv = pd.read_csv("./cleaned_data/gmv_clean.csv", encoding='utf-8-sig')


phone_stats = (
    df_gmv.groupby("Phone_clean")
    .size()
    .reset_index(name="Count")
)

phone_dup = phone_stats[phone_stats["Count"] > 1]

print(f"Phone unique: {phone_stats.shape[0]:,}")
print(f"Duplicate phones: {phone_dup.shape[0]:,}")
print(f"Rows involved: {phone_dup['Count'].sum():,}")
print(phone_dup.sort_values("Count", ascending=False).head(20))

Phone unique: 457
Duplicate phones: 6
Rows involved: 12
      Phone_clean  Count
83    84865800589      2
202   84943111987      2
204   84944022155      2
355  491631303303      2
381  818033281246      2
428  821040844054      2


In [80]:
uid_stats = (
    df_gmv.groupby("UID_clean")
    .size()
    .reset_index(name="Count")
)

uid_dup = uid_stats[uid_stats["Count"] > 1]

print(f"UID unique: {uid_stats.shape[0]:,}")
print(f"Duplicate UIDs: {uid_dup.shape[0]:,}")
print(f"Rows involved: {uid_dup['Count'].sum():,}")
print(uid_dup.sort_values("Count", ascending=False).head(20))

UID unique: 454
Duplicate UIDs: 9
Rows involved: 18
      UID_clean  Count
98   3309549687      2
110  3310023176      2
139  3310871114      2
318  3311684368      2
326  3311716270      2
334  3311738141      2
359  3311861764      2
446  3312234999      2
451  3312285639      2


In [81]:
pair_stats = (
    df_gmv.groupby(["Phone_clean", "UID_clean"])
    .size()
    .reset_index(name="Count")
)

pair_dup = pair_stats[pair_stats["Count"] > 1]

print(f"Unique pairs: {pair_stats.shape[0]:,}")
print(f"Duplicate pairs: {pair_dup.shape[0]:,}")
print(f"Rows involved: {pair_dup['Count'].sum():,}")
print(pair_dup.sort_values("Count", ascending=False).head(20))

Unique pairs: 457
Duplicate pairs: 6
Rows involved: 12
      Phone_clean   UID_clean  Count
83    84865800589  3309549687      2
202   84943111987  3312285639      2
204   84944022155  3311861764      2
355  491631303303  3310871114      2
381  818033281246  3311738141      2
428  821040844054  3311716270      2


In [82]:
pair_stats = (
    df_gmv.groupby(["Phone_clean", "UID_clean"])
    .size()
    .reset_index(name="Count")
)

# Các cặp xuất hiện đúng 1 lần
pair_unique = pair_stats[pair_stats["Count"] == 1]

print(f"Total GMV rows           : {len(df_gmv):,}")
print(f"Distinct Phone+UID pairs : {len(pair_stats):,}")
print(f"Unique Phone+UID pairs   : {len(pair_unique):,}")
print(f"Unique rate              : {len(pair_unique) / len(pair_stats):.2%}")

Total GMV rows           : 463
Distinct Phone+UID pairs : 457
Unique Phone+UID pairs   : 451
Unique rate              : 98.69%


### Tìm cách match

In [83]:
df_gmv = pd.read_csv(
    "./cleaned_data/gmv_clean.csv",
    dtype={
        "Phone_clean": str,
        "UID_clean": str
    },
    encoding="utf-8-sig"
)

df_leads = pd.read_csv(
    "./cleaned_data/leads_clean.csv",
    dtype={
        "Phone_clean": str,
        "UID_clean": str
    },
    encoding="utf-8-sig"
)

In [92]:
# Extract unique combinations of Phone and UID into a new DataFrame
unique_gmv_keys_df = df_gmv[["Phone_clean", "UID_clean"]].drop_duplicates()

# Reset index to make it clean
unique_gmv_keys_df = unique_gmv_keys_df.reset_index(drop=True)

print(f"Tổng số cặp key duy nhất trong GMV: {len(unique_gmv_keys_df)}")
print(unique_gmv_keys_df.head())

Tổng số cặp key duy nhất trong GMV: 457
   Phone_clean   UID_clean
0  84908224672  3311001921
1  84909517679  3179205818
2  84964678701  3311304274
3  84389970625  3310902627
4  84938593961  3309271098


In [93]:
# Lọc ra các cặp Phone và UID không trùng lặp từ Leads
unique_leads_keys_df = df_leads[["Phone_clean", "UID_clean"]].drop_duplicates()

# Reset lại index cho gọn gàng
unique_leads_keys_df = unique_leads_keys_df.reset_index(drop=True)

# Loại bỏ những dòng mà cả Phone và UID đều rỗng (nếu có)
unique_leads_keys_df = unique_leads_keys_df[
    (unique_leads_keys_df["Phone_clean"] != "") | 
    (unique_leads_keys_df["UID_clean"] != "")
]

print(f"Tổng số cặp key duy nhất trong Leads: {len(unique_leads_keys_df)}")
print(unique_leads_keys_df.head())

Tổng số cặp key duy nhất trong Leads: 15829
   Phone_clean   UID_clean
0          NaN         NaN
1  84919249237  3310947243
2  84965468525  3310952712
3  84985951781  3293618952
4  84972856556  3310947249


In [96]:
# Tìm tập giao (intersection) giữa 2 bảng dựa trên cả 2 cột
chung_ca_hai = pd.merge(unique_gmv_keys_df, unique_leads_keys_df, on=["Phone_clean", "UID_clean"], how="inner")

print(f"Số lượng khách hàng khớp uid + phone: {len(chung_ca_hai)}")

Số lượng khách hàng khớp uid + phone: 259


In [ ]:
# 1. Tạo Set cho GMV (Tự động lọc unique)
set_gmv = set(zip(df_gmv["Phone_clean"], df_gmv["UID_clean"]))

# 2. Tạo Set cho Leads 
set_leads = set(zip(df_leads["Phone_clean"], df_leads["UID_clean"]))

# 3. Loại bỏ rác: Xóa cặp rỗng (khách không có cả phone lẫn UID) để tránh match sai
set_gmv.discard(("", ""))
set_leads.discard(("", ""))

# 4. Tìm Giao điểm (Intersection): Những cặp vừa có trong GMV, vừa có trong Leads
matched_keys = set_gmv.intersection(set_leads)

print(f"Số cặp unique trong GMV: {len(set_gmv)}")
print(f"Số cặp unique trong Leads: {len(set_leads)}")
print("-" * 40)
print(f"TỔNG SỐ CẶP MATCH THÀNH CÔNG: {len(matched_keys)}")

# Nếu muốn xem thử 5 cặp match được:
print("\nVí dụ 5 cặp match được (Phone, UID):")
print(list(matched_keys)[:5])

Số cặp unique trong GMV: 457
Số cặp unique trong Leads: 15829
----------------------------------------
TỔNG SỐ CẶP MATCH THÀNH CÔNG: 259

Ví dụ 5 cặp match được (Phone, UID):
[('84982135774', '3311850607'), ('84986694373', '3311939526'), ('84908655015', '3311572089'), ('818042129159', '3311534573'), ('84937973897', '3302383080')]


In [ ]:
# Assuming matched_keys is the set of tuples from the previous step:
# matched_keys = set_gmv.intersection(set_leads)

# 1. Create a boolean mask to identify which rows in GMV match the keys
is_already_matched = df_gmv.set_index(["Phone_clean", "UID_clean"]).index.isin(matched_keys)

# 2. Filter out the matched ones using the ~ (NOT) operator
unmatched_gmv = df_gmv[~is_already_matched]


matched_gmv = df_gmv[is_already_matched]

# Print the results to verify the split
print(f"Total original GMV rows: {len(df_gmv)}")
print(f"Matched GMV rows (Removed): {len(matched_gmv)}")
print(f"Remaining Unmatched GMV rows: {len(unmatched_gmv)}")

Total original GMV rows: 463
Matched GMV rows (Removed): 261
Remaining Unmatched GMV rows: 202


In [105]:
# 1. Tạo boolean mask để xác định những dòng trong Leads đã khớp với matched_keys
is_already_matched_leads = df_leads.set_index(["Phone_clean", "UID_clean"]).index.isin(matched_keys)

# 2. Lọc lấy những dòng CHƯA khớp (dùng dấu ~ để đảo ngược điều kiện)
unmatched_leads = df_leads[~is_already_matched_leads]

# Lấy ra những dòng ĐÃ khớp (tùy chọn, để kiểm tra chéo nếu cần)
matched_leads = df_leads[is_already_matched_leads]

# In kết quả để kiểm tra số lượng
print(f"Total original Leads rows: {len(df_leads)}")
print(f"Matched Leads rows (Removed): {len(matched_leads)}")
print(f"Remaining Unmatched GMV rows:: {len(unmatched_leads)}")

Total original Leads rows: 16336
Matched Leads rows (Removed): 267
Remaining Unmatched GMV rows:: 16069


In [119]:
print(matched_leads)

      Đầu Ngày           Họ và tên Người trực  Số điện thoại      Tuổi  \
16       01/05    Thuy Minh Nguyen       Linh   84-915272524  7 và 10t   
63       01/05           Dương Mon      Quỳnh   84-962340965  3-5 tuổi   
75       02/05      Một Đời An Yên       Liên   84-344668676     6-10t   
105      02/05       Miki Tiuganji        NaN  81-9013302406       12t   
106      02/05     Thanh nguyenthi        NaN  81-7021914402    8 tuổi   
...        ...                 ...        ...            ...       ...   
14273    16/01          Nhung Trần         Ly   84-981442192        6t   
15261    24/01         Thu Ha Thai         Ly   84-935501409        5t   
15479    26/01             Hà Hồng         Ly   84-936141588       NaN   
15489    26/01  Nguyen Huyen Huyen         Ly   84-964675729       NaN   
15744    28/01            Vanha Tn         Ly   84-902075539       NaN   

       Quốc Gia                                               Note  \
16     Việt Nam                          

In [103]:
print(unmatched_leads)

      Đầu Ngày      Họ và tên Người trực Số điện thoại     Tuổi  Quốc Gia  \
0          NaN            NaN        NaN           NaN      NaN       NaN   
1        01/05     Huỳnh Uyên       Liên  84-919249237  8 + 12t  Việt Nam   
2        01/05          Ni Ni       Liên  84-965468525     3-5t  Việt Nam   
3        01/05  Phương Phương       Liên  84-985951781    6-10t  Việt Nam   
4        01/05      Hoàng Huế       Liên  84-972856556      NaN  Việt Nam   
...        ...            ...        ...           ...      ...       ...   
16331    31/01          Ok là        NaN  84-909693803      NaN  Việt Nam   
16332    31/01     hahaha1999        NaN  84-986806948      NaN  Việt Nam   
16333    31/01   Trần Thị Sen        NaN  84-981971660      NaN  Việt Nam   
16334    31/01        Thảo Đỗ        NaN  84-389479460      NaN  Việt Nam   
16335    31/01      Trần Dung        NaN  84-935030790      NaN  Việt Nam   

                                                    Note           TVTS  \


In [104]:
print(unmatched_gmv)

      bank day bank time       Gateway           User Name          Phone  \
0     2026/5/2     18:28       VP Bank    C Thanh - Bé Hân   84-908224672   
1    2026/5/11       NaN           NaN  Chị Nga - Bé Khang   84-909517679   
3    2026/5/14     09:30      Agribank    Chị Huệ - Bé Thi   84-389970625   
6    2026/5/20       NaN           NaN  C Khuê - Huy Hoàng   84-978827804   
9    2026/5/24     20:21       MB Bank    C Dung - Bé Diệp   84-938222653   
..         ...       ...           ...                 ...            ...   
452  2026/5/31     20:58       Shinhan           Đăng Khoa   84-368776525   
458  2026/5/31     21:40  VCB Digibank                  Bé  81-7035351368   
460  2026/5/31         x          VIMO                 Kem   84-969663003   
461  2026/5/31     23:38             x           Thiên Kim   84-943111987   
462  2026/5/31     23:38        MBBank                 Mon   84-943111987   

            UID                              Package Fixed/ non-fixed  \
0 

In [110]:
# 1. Lọc lấy các UID hợp lệ (khác rỗng) và ném vào Set để tự động xóa trùng
set_uid_gmv = set(unmatched_gmv[unmatched_gmv["UID_clean"] != ""]["UID_clean"])
set_uid_leads = set(unmatched_leads[unmatched_leads["UID_clean"] != ""]["UID_clean"])

# 2. Tìm giao điểm (intersection)
uid_chung = set_uid_gmv.intersection(set_uid_leads)

# 3. In kết quả thăm dò
print(f"Số UID duy nhất trong tập GMV còn lại: {len(set_uid_gmv)}")
print(f"Số UID duy nhất trong tập Leads còn lại: {len(set_uid_leads)}")
print("-" * 40)

if len(uid_chung) > 0:
    print(f"CÓ TỒN TẠI! Phát hiện {len(uid_chung)} UID xuất hiện ở cả 2 bên.")
    print("Ví dụ 5 UID chung đầu tiên:")
    print(list(uid_chung)[:5])
else:
    print("KHÔNG CÓ UID NÀO CHUNG")

Số UID duy nhất trong tập GMV còn lại: 195
Số UID duy nhất trong tập Leads còn lại: 15018
----------------------------------------
KHÔNG CÓ UID NÀO CHUNG


In [117]:
print(set_uid_gmv)

{'3311696395', '3288489392', '3312256044', '3312087224', '3311188064', '3311719439', '3312123609', '3311812212', '3303835667', '3270105568', '3311494013', '3310177620', '3311118484', '3312266413', '3180717063', '3299808858', '3309875453', '3312225025', '3311837335', '3311781820', '3311538332', '3302670521', '3293995784', '3312150282', '3311496384', '3311738141', '3311508463', '3292493174', '3311804423', '3311924661', '3311228095', '3310486815', '3302064321', '3311237585', '3312306410', '3312248783', '3312097348', '3311996848', '3311469735', '3311450402', '3311376353', '3297314626', '3311490911', '3311153969', '3312015912', '3311356664', '3311992414', '3311951330', '3311354979', '3310673494', '3310314777', '3311688733', '3311993190', '3311501540', '3312006640', '3311803674', '3311414144', '3310902627', '3311784220', '3311649925', '3290550889', '3280669999', '3312125506', '3312091945', '3312015019', '3312134608', '3311717384', '3311229312', '3311506785', '3311032514', '3311114137', '3311

In [111]:
# Extract valid phones (not empty) and convert to sets to remove duplicates
set_phone_gmv = set(unmatched_gmv[unmatched_gmv["Phone_clean"] != ""]["Phone_clean"])
set_phone_leads = set(unmatched_leads[unmatched_leads["Phone_clean"] != ""]["Phone_clean"])

# Find the intersection of both sets
phone_chung = set_phone_gmv.intersection(set_phone_leads)

# Print the exploration results
print(f"Số lượng Phone duy nhất trong tập GMV còn lại: {len(set_phone_gmv)}")
print(f"Số lượng Phone duy nhất trong tập Leads còn lại: {len(set_phone_leads)}")
print("-" * 40)

if len(phone_chung) > 0:
    print(f"CÓ TỒN TẠI! Phát hiện {len(phone_chung)} Số điện thoại xuất hiện ở cả 2 bên.")
    print("Ví dụ 5 Số điện thoại chung đầu tiên:")
    print(list(phone_chung)[:5])
else:
    print("KHÔNG CÓ SỐ ĐIỆN THOẠI NÀO CHUNG")

Số lượng Phone duy nhất trong tập GMV còn lại: 198
Số lượng Phone duy nhất trong tập Leads còn lại: 15419
----------------------------------------
CÓ TỒN TẠI! Phát hiện 19 Số điện thoại xuất hiện ở cả 2 bên.
Ví dụ 5 Số điện thoại chung đầu tiên:
['817084801084', '84395758301', '84917399090', '817090779689', '420773055288']


In [113]:
# Filter out empty phone numbers to avoid matching blanks
valid_phone_gmv = unmatched_gmv[unmatched_gmv["Phone_clean"] != ""]
valid_phone_leads = unmatched_leads[unmatched_leads["Phone_clean"] != ""]

# CRITICAL: Deduplicate leads by Phone_clean to maintain 1-1 relationship and protect GMV
unique_phone_leads = valid_phone_leads.drop_duplicates(subset=["Phone_clean"], keep="first")

# Perform Inner Join on Phone_clean
# Pandas will automatically add suffixes to overlapping columns like UID_clean
matched_by_phone = pd.merge(
    valid_phone_gmv,
    unique_phone_leads,
    on="Phone_clean",
    how="inner",
    suffixes=("_gmv", "_lead")
)

# Filter rows where UID from GMV does not match UID from Leads
# We also make sure we are not comparing empty strings
same_phone_diff_uid = matched_by_phone[
    (matched_by_phone["UID_clean_gmv"] != matched_by_phone["UID_clean_lead"]) &
    (matched_by_phone["UID_clean_gmv"] != "") &
    (matched_by_phone["UID_clean_lead"] != "")
]

# Print the final report
print(f"Tổng số đơn GMV cứu lại được nhờ Join theo Phone: {len(matched_by_phone)}")
print(f"Số đơn có CÙNG Phone nhưng KHÁC UID: {len(same_phone_diff_uid)}")
print("-" * 50)

if len(same_phone_diff_uid) > 0:
    print("Danh sách chi tiết các trường hợp (Cùng SĐT - Khác UID):")
    # Select only relevant columns to display clearly
    display_cols = ["Phone_clean", "UID_clean_gmv", "UID_clean_lead"]
    print(same_phone_diff_uid[display_cols])
else:
    print("Không có trường hợp nào khác UID (Hoặc UID bị trống ở 1 trong 2 bên).")

Tổng số đơn GMV cứu lại được nhờ Join theo Phone: 19
Số đơn có CÙNG Phone nhưng KHÁC UID: 19
--------------------------------------------------
Danh sách chi tiết các trường hợp (Cùng SĐT - Khác UID):
     Phone_clean UID_clean_gmv UID_clean_lead
0    84978827804    3310167280   84-978827804
1   819042075465    3310673494     3310547595
2   817090779689    3300978231            NaN
3   817084801084    3311228463            NaN
4   821034881395    3230291710            NaN
5   420773055288    3286214118            NaN
6   817014007482    3308967220            NaN
7    84395758301    3311880675            NaN
8   491727155802    3311837335            NaN
9   817090282741    3305714444            NaN
10   84979153635    3198416338            NaN
11   84917399090    3312048610            NaN
12  817090293668    3311228095     3311193217
13  818090801510    3312015019     3311905251
14  818023794606    3311996181            NaN
15  818058438726    3312134608     3312003759
16   84357978861 

In [114]:
# Extract the unique phone numbers that were successfully matched in Tier 3
matched_phones = matched_by_phone["Phone_clean"].unique()

# Filter out the matched rows to find the final remaining organic GMV
final_unmatched_gmv = unmatched_gmv[~unmatched_gmv["Phone_clean"].isin(matched_phones)]

# Print the final attribution summary
print(f"Số đơn GMV trước khi ghép theo Phone: {len(unmatched_gmv)}")
print(f"Số đơn đã ghép thành công (Attributed to Leads): {len(matched_by_phone)}")
print("-" * 50)
print(f" TỔNG SỐ ĐƠN GMV CÒN LẠI CUỐI CÙNG (Organic/Không rõ nguồn): {len(final_unmatched_gmv)}")

# Optional: View a sample of the remaining un-attributable data
if len(final_unmatched_gmv) > 0:
    print("\nVí dụ vài đơn không thể ghép nối với Leads:")
    print(final_unmatched_gmv[["Phone_clean", "UID_clean"]].head())

Số đơn GMV trước khi ghép theo Phone: 202
Số đơn đã ghép thành công (Attributed to Leads): 19
--------------------------------------------------
 TỔNG SỐ ĐƠN GMV CÒN LẠI CUỐI CÙNG (Organic/Không rõ nguồn): 183

Ví dụ vài đơn không thể ghép nối với Leads:
    Phone_clean   UID_clean
0   84908224672  3311001921
1   84909517679  3179205818
3   84389970625  3310902627
9   84938222653  3311951330
10  84844534222  3311919278


In [ ]:
# Extract columns from both dataframes as lists
gmv_cols = df_gmv.columns.tolist()
leads_cols = df_leads.columns.tolist()

# Find common columns that exist in both dataframes (potential join keys)
common_cols = list(set(gmv_cols).intersection(set(leads_cols)))

# Print GMV columns line by line for easy reading
print(f"TỔNG SỐ CỘT GMV: {len(gmv_cols)}")
for i, col in enumerate(gmv_cols, 1):
    print(f"  {i}. {col}")

# Print Leads columns line by line
print(f"\nTỔNG SỐ CỘT LEADS: {len(leads_cols)}")
for i, col in enumerate(leads_cols, 1):
    print(f"  {i}. {col}")

# Print the automatically matched columns
print("\nCÁC CỘT CÓ THỂ DÙNG ĐỂ JOIN (TRÙNG TÊN):")
if common_cols:
    for col in common_cols:
        print(f"  => {col}")
else:
    print("Không có cột nào trùng tên hoàn toàn (Cần tự đối chiếu bằng mắt).")

TỔNG SỐ CỘT GMV: 38
  1. bank day
  2. bank time
  3. Gateway
  4. User Name
  5. Phone
  6. UID
  7. Package
  8. Fixed/ non-fixed
  9. Pay Time
  10. Real Pay(VND)
  11. GMV (RMB)
  12. Payment Method
  13. Type
  14. Sales
  15. Note
  16. Month of payment
  17. Unnamed: 16
  18. Full Price(VND)
  19. 采购价格 
（套餐配置）
  20. 渠道号
  21. 首次申请试听日期
  22. 首次申请试听渠道号
  23. 首次购课时间
  24. Source
  25. 已批工单
  26. 总 B (被推荐） 课数
  27. A (推荐人） 课数 (送PH)
  28. A
  29. A (推荐人）课数 (送UK)
  30. B
  31. 实际卖课单价 VND
  32. 实际卖课单价 RMB
  33. PH/UK
  34. 采购价格（包括转介绍赠送课)
  35. TEAM
  36. Nguồn
  37. UID_clean
  38. Phone_clean

TỔNG SỐ CỘT LEADS: 28
  1. Đầu Ngày
  2. Họ và tên
  3. Người trực
  4. Số điện thoại
  5. Tuổi
  6. Quốc Gia
  7. Note
  8. TVTS
  9. Ad_id
  10. Nguồn
  11. Quốc gia 
Quảng cáo
  12. Mẫu Quảng cáo
  13. Mã CRM
  14. Sale sau phân
  15. Link FB
  16. UID
  17. TT
  18. Trạng thái
  19. Ngày lên học thử
  20. Current Binded Sale
  21. Sales in latest system-weighted allocation
  22. Latest syste

In [124]:
df_gmv = pd.read_csv("./cleaned_data/SM_HN_HCM_REV_filtered_2026_5.csv", encoding='utf-8-sig')
# Create boolean masks to identify missing or empty values in GMV
# We check for both NaN (null) and empty strings ("") just to be safe
mask_no_phone = df_gmv["Phone"].isna() | (df_gmv["Phone"] == "")
mask_no_uid = df_gmv["UID"].isna() | (df_gmv["UID"] == "")

# Isolate the exact scenarios to avoid double counting
missing_both = mask_no_phone & mask_no_uid
missing_phone_only = mask_no_phone & ~mask_no_uid
missing_uid_only = mask_no_uid & ~mask_no_phone

# Print the final missing data report
print("--- THỐNG KÊ DỮ LIỆU BỊ HỎNG/THIẾU TRONG GMV ---")
print(f"Tổng số dòng GMV hiện tại: {len(df_gmv)}")
print("-" * 50)
print(f"1. Số dòng KHÔNG CÓ CẢ 2 (No Phone & No UID): {missing_both.sum()}")
print(f"2. Số dòng CHỈ THIẾU SĐT (Has UID, but no Phone): {missing_phone_only.sum()}")
print(f"3. Số dòng CHỈ THIẾU UID (Has Phone, but no UID): {missing_uid_only.sum()}")
print("-" * 50)

# Total rows that are missing at least one piece of information
total_missing_at_least_one = (mask_no_phone | mask_no_uid).sum()
print(f"TỔNG SỐ DÒNG BỊ THIẾU ÍT NHẤT 1 TRƯỜNG: {total_missing_at_least_one}")

--- THỐNG KÊ DỮ LIỆU BỊ HỎNG/THIẾU TRONG GMV ---
Tổng số dòng GMV hiện tại: 463
--------------------------------------------------
1. Số dòng KHÔNG CÓ CẢ 2 (No Phone & No UID): 0
2. Số dòng CHỈ THIẾU SĐT (Has UID, but no Phone): 0
3. Số dòng CHỈ THIẾU UID (Has Phone, but no UID): 0
--------------------------------------------------
TỔNG SỐ DÒNG BỊ THIẾU ÍT NHẤT 1 TRƯỜNG: 0
